In [11]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle

In [12]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

Database config loaded: 100.75.213.18
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [13]:
with open("fase_3_afrida.pkl", "rb") as f:
    fase_3_afrida = pickle.load(f)
    
print("📦 Isi file afrida:")
for key in fase_3_afrida.keys():
    print(f" - {key}: {fase_3_afrida[key].shape}")

📦 Isi file afrida:
 - sop_kategori: (3, 2)
 - sop: (4, 4)
 - surat_keluar: (213, 1)
 - verifikasi_surat_keluar: (477, 4)
 - surat_tugas: (135, 1)
 - surat_tugas_anggota: (304, 2)


In [14]:
# Function insert
def insert_data_to_database(db_connection, cursor, tables_data, tables_to_insert):
    """
    Insert data ke database baru dari dictionary dataframes
    """
    results = {}
    
    print("="*80)
    print("MEMULAI INSERT DATA KE DATABASE BARU")
    print("="*80)
    
    for table_name in tables_to_insert:
        try:
            if table_name not in tables_data:
                print(f"\n⚠️  {table_name}: Tidak ditemukan di data, skip")
                results[table_name] = {'status': 'skipped', 'rows': 0}
                continue
            
            df_to_insert = tables_data[table_name]
            
            if df_to_insert.empty:
                print(f"\n⚠️  {table_name}: DataFrame kosong, skip insert")
                results[table_name] = {'status': 'empty', 'rows': 0}
                continue
            
            df_to_insert = df_to_insert.dropna(axis=1, how='all')
            
            columns = ', '.join([f'`{col}`' for col in df_to_insert.columns])
            placeholders = ', '.join(['%s'] * len(df_to_insert.columns))
            
            insert_query = f"INSERT INTO `{table_name}` ({columns}) VALUES ({placeholders})"
            data_to_insert = [tuple(row) for row in df_to_insert.values]
            
            cursor.executemany(insert_query, data_to_insert)
            db_connection.commit()
            
            print(f"✓ {table_name}: Berhasil insert {len(data_to_insert)} baris")
            results[table_name] = {'status': 'success', 'rows': len(data_to_insert)}
            
        except Exception as e:
            db_connection.rollback()
            print(f"✗ {table_name}: Gagal insert - {e}")
            results[table_name] = {'status': 'failed', 'error': str(e), 'rows': 0}
    
    print("\n" + "="*80)
    print("PROSES INSERT SELESAI")
    print("="*80)
    
    return results
    
# Merge the two dictionaries
# tables_data = {**fase_2_afrida, **fase_2_cimut}
tables_to_insert = ['sop_kategori', 'sop',  'surat_keluar', 'verifikasi_surat_keluar', 'surat_tugas', 'surat_tugas_anggota']
# tables_to_insert += ['users', 'divisions', 'shift_kerja', 'admin_sarpras', 'sop_kategori']
results = insert_data_to_database(db_new, cursor_new, fase_3_afrida, tables_to_insert)

MEMULAI INSERT DATA KE DATABASE BARU
✓ sop_kategori: Berhasil insert 3 baris
✓ sop: Berhasil insert 4 baris
✗ surat_keluar: Gagal insert - 1054 (42S22): Unknown column 'nosurat' in 'field list'
✗ verifikasi_surat_keluar: Gagal insert - Failed executing the operation; Python type NAType cannot be converted
✗ surat_tugas: Gagal insert - 1054 (42S22): Unknown column 'lokasi' in 'field list'
✗ surat_tugas_anggota: Gagal insert - 1452 (23000): Cannot add or update a child row: a foreign key constraint fails (`dataleap_v5_migration`.`surat_tugas_anggota`, CONSTRAINT `surat_tugas_anggota_id_st_foreign` FOREIGN KEY (`id_st`) REFERENCES `surat_tugas` (`id_st`) ON DELETE SET NULL)

PROSES INSERT SELESAI


In [10]:
# Truncate tables yang sudah diinsert (dengan force delete)
tables_to_truncate = tables_to_insert

print("="*80)
print("MEMULAI TRUNCATE DATA DI DATABASE BARU")
print("="*80)

# Disable foreign key checks
cursor_new.execute("SET FOREIGN_KEY_CHECKS=0")
db_new.commit()

for sop in tables_to_truncate:
    try:
        truncate_query = f"TRUNCATE TABLE `{sop}`"
        cursor_new.execute(truncate_query)
        db_new.commit()
        print(f"✓ {sop}: Berhasil truncate")
    except Exception as e:
        db_new.rollback()
        print(f"✗ {sop}: Gagal truncate - {e}")

# Re-enable foreign key checks
cursor_new.execute("SET FOREIGN_KEY_CHECKS=1")
db_new.commit()

print("\n" + "="*80)
print("PROSES TRUNCATE SELESAI")
print("="*80)

MEMULAI TRUNCATE DATA DI DATABASE BARU
✓ sop_kategori: Berhasil truncate
✓ sop: Berhasil truncate
✓ surat_keluar: Berhasil truncate
✓ verifikasi_surat_keluar: Berhasil truncate
✓ surat_tugas: Berhasil truncate
✓ surat_tugas_anggota: Berhasil truncate

PROSES TRUNCATE SELESAI
